In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm 
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split

In [2]:
random_state = 42

## Credit Card Dataset

In [3]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/credit_card_cleaned.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


First lets undersample since this is a fraud dataset

In [4]:
X = df.drop(columns='Class')

In [5]:
y = df['Class']

In [6]:
print("Original dataset shape:", Counter(y))

Original dataset shape: Counter({0: 284315, 1: 492})


In [7]:
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Resampled dataset shape: Counter({0: 4920, 1: 492})


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

In [9]:
model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train, y_train) 

weights_df = pd.DataFrame({
    'Feature': X.columns,
    'Weight (Coefficient)': model.coef_[0]
}).sort_values(by='Weight (Coefficient)', ascending=False)
print(weights_df) 

   Feature  Weight (Coefficient)
4       V4              0.953849
22     V22              0.850403
1       V1              0.400199
24     V24              0.366729
28     V28              0.352476
21     V21              0.287625
11     V11              0.173203
7       V7              0.098008
26     V26              0.037010
5       V5              0.000218
0     Time             -0.000020
29  Amount             -0.001001
27     V27             -0.030545
18     V18             -0.040189
19     V19             -0.101945
15     V15             -0.121067
6       V6             -0.149903
20     V20             -0.165499
9       V9             -0.200278
3       V3             -0.297376
23     V23             -0.317203
13     V13             -0.330462
2       V2             -0.474043
8       V8             -0.581539
16     V16             -0.613624
17     V17             -0.676820
12     V12             -0.759882
10     V10             -0.866444
25     V25             -0.932644
14     V14

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
y_pred = results.predict(X_test)

In [11]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.8406375507043135

In [12]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

In [13]:
vif_data

,Feature,VIF
0,Time,2.339084
1,V1,1.621694
2,V2,3.869377
3,V3,1.255585
4,V4,1.137944
5,V5,2.753075
6,V6,1.522122
7,V7,2.510165
8,V8,1.097151
9,V9,1.018831


The model overpredicts a lot of negative cases, and underpredicts a lot of positive cases. There are likely non-linear patterns in the data

There is not too much multicolinearity, aside from Amount 

Attempting Polynomial Terms

In [14]:
df_X_train_with_square = pd.concat((X_train, (X_train**2).rename(columns = {f"{var}": f"{var}_2" for var in X.columns})), axis = 1) 
df_X_test_with_square = pd.concat((X_test, (X_test**2).rename(columns = {f"{var}": f"{var}_2" for var in X.columns})), axis = 1) 

model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(df_X_train_with_square, y_train) 

In [15]:
df_X_train_with_square

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20_2,V21_2,V22_2,V23_2,V24_2,V25_2,V26_2,V27_2,V28_2,Amount_2
264852,161625.0,1.835371,-0.078636,-1.805991,0.608965,0.161693,-1.379083,0.470875,-0.452902,0.433563,...,0.005335,0.078042,0.614763,0.015019,0.002061,0.052694,0.012836,0.000194,0.000108,13016.5281
113745,73187.0,-0.481811,1.008502,1.674705,-0.004484,-0.037582,-0.613361,0.640558,0.046928,-0.432810,...,0.003787,0.031476,0.171629,0.002421,0.146611,0.094594,0.006466,0.079726,0.014678,12.7449
42573,41157.0,1.211835,0.694404,-0.436292,0.890024,0.234105,-1.149547,0.412392,-0.261166,-0.355015,...,0.001946,0.008571,0.015337,0.000696,0.100870,0.240655,0.140485,0.000046,0.002031,0.5776
76132,56410.0,1.179266,0.351984,0.928646,1.181232,-0.632445,-0.924033,0.009343,-0.192170,0.000758,...,0.003813,0.023919,0.167952,0.018266,0.479190,0.084332,0.419031,0.001919,0.001694,99.8001
247242,153516.0,-0.836723,-0.046779,1.884473,-2.021349,-1.353374,0.211389,-1.066001,0.876512,-0.115364,...,0.001157,0.071125,0.835460,0.016382,0.001362,0.098591,0.093026,0.091752,0.020374,5.2900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48858,43815.0,1.387618,-0.525932,0.190953,-0.555127,-1.111076,-1.173276,-0.386324,-0.222954,-0.751894,...,0.289162,0.156891,0.535517,0.002613,0.141992,0.053509,1.090126,0.005655,0.000012,283.2489
126236,77966.0,1.045365,-1.737480,-0.030130,-0.920359,-1.793838,-1.163170,-0.404206,-0.356336,-1.759074,...,0.006237,0.002643,0.042808,0.060929,0.528095,0.273890,0.011581,0.001091,0.002783,64009.0000
150697,93920.0,-12.381048,8.213022,-16.962530,7.116091,-9.772826,-3.666836,-16.147363,2.078706,-4.250657,...,0.289223,0.028124,2.260252,0.589448,0.138348,2.004035,0.267312,0.188895,0.085686,9409.0000
99524,67157.0,-1.723162,1.476182,0.803252,1.490206,-1.167526,0.438987,-0.736028,1.362561,-0.303930,...,0.118867,0.034877,0.284221,0.000003,0.022568,0.011668,0.049738,0.136651,0.011532,1099.5856


In [16]:
y_pred = results.predict(df_X_test_with_square)

In [17]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.0904893813481071

This model is actually a bit worse

In [18]:
vif_data = pd.DataFrame()
vif_data["Feature"] = df_X_train_with_square.columns
vif_data["VIF"] = [variance_inflation_factor(df_X_train_with_square.values, i) for i in range(df_X_train_with_square.shape[1])]

### Conclusions

In the OLS model, we only see Amount with severe multicollinearity (12.116701) through VIF. Next is V2 with 4.422390 VIF. I calculate the PR-AUC as 0.0033740975340107947

Next I performed polynomial interaction terms, doubling from 31 to 62 features. I calculate the MSE as 0.005065552998079394. When checking for VIF, we see 21 features with VIF > 5, so I dropped them, leaving 41 features total. Some of the independent variables highly correlate with each other, making it difficult for OLS to isolate the individual effect of each predictor.

Now I run OLS again on the 41 features and get a MSE of 0.006337391222183026

As we can see, the model did not improve with more features, simpler was better. The polynomial terms added noise. Also, removing the multicollinear features degraded performance further. This could be due to the polynomial terms introducing overfitting or highly unstable coefficient estimates.

To handle the multicollinearity, we can try regularization techniques like Ridge

## IBM Dataset

### Read Data and Preliminary Cleaning

In [19]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [20]:
cols_to_drop = [c for c in df.columns if "Mean" in c or "Median" in c]
cols_to_drop.extend(['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'])
df.drop(columns=cols_to_drop, inplace=True)

In [21]:
df

,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,Receiving Currency_Canadian Dollar,Receiving Currency_Euro,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,3697.340000,3697.340000,0,3697.340000,3697.340000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,0.010000,0.010000,0,0.010000,0.010000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,14675.570000,14675.570000,0,14675.570000,14675.570000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2806.970000,2806.970000,0,2806.970000,2806.970000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,36682.970000,36682.970000,0,36682.970000,36682.970000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,0.154978,0.154978,0,3107.386389,3107.386389,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,0.108128,0.108128,0,2168.020464,2168.020464,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,0.004988,0.004988,0,100.011894,100.011894,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,0.038417,0.038417,0,770.280058,770.280058,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [22]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor


def clean_data(df_input):
    # 1. Create a copy to avoid modifying original data
    df_new = df_input.copy()

    # Keeping both standard Amount Received/Paid + Amount Received/Paid USD was causing multicollinearity, we remove non USD values
    cols_to_drop = ["Amount Received", "Amount Paid"]
    df_new = df_new.drop(columns=[c for c in cols_to_drop if c in df_new.columns])

    # Fix Dummy Variable Trap: Drop one column from each categorical group
    # We identify groups by splitting column names on the underscore '_'
    dummy_groups = [
        "Receiving Currency",
        "Payment Currency",
        "Payment Format",
    ]

    for group in dummy_groups:
        # Find all columns belonging to this specific categorical group
        group_cols = [c for c in df_new.columns if c.startswith(group)]
        if len(group_cols) > 1:
            # Drop the very first column of this group to serve as the baseline
            df_new = df_new.drop(columns=[group_cols[0]])
            print(f"Dropped baseline category to fix 'inf': {group_cols[0]}")

    # 4. Handle Constant Column (Ensure only one exists and it has variance)
    if "const" in df_new.columns:
        # If const is all zeros or broken from scaling, reset it
        df_new = df_new.drop(columns=["const"])

    df_new.to_csv('ibm_hi_small_trans_cleaned_semester3.csv', index=False)

    # # 5. Calculate clean VIF Data
    # vif_data = pd.DataFrame()
    # vif_data["Feature"] = X.columns
    # vif_data["VIF"] = [
    #     variance_inflation_factor(X.values, i) for i in range(X.shape[1])
    # ]

    # return vif_data, X

clean_data(df)

Dropped baseline category to fix 'inf': Receiving Currency_Australian Dollar
Dropped baseline category to fix 'inf': Payment Currency_Australian Dollar
Dropped baseline category to fix 'inf': Payment Format_ACH


In [23]:
df = pd.read_csv('ibm_hi_small_trans_cleaned_semester3.csv')
df

,Is Laundering,Amount_Received_USD,Amount_Paid_USD,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,Receiving Currency_Canadian Dollar,Receiving Currency_Euro,Receiving Currency_Mexican Peso,Receiving Currency_Ruble,Receiving Currency_Rupee,...,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,0,3697.340000,3697.340000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,0,0.010000,0.010000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,0,14675.570000,14675.570000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,0,2806.970000,2806.970000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,0,36682.970000,36682.970000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,0,3107.386389,3107.386389,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,0,2168.020464,2168.020464,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,0,100.011894,100.011894,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,0,770.280058,770.280058,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


### Scaling

First lets undersample since this is a fraud dataset

In [24]:
X = df.drop(columns='Is Laundering')

In [25]:
y = df['Is Laundering']

In [26]:
print("Original dataset shape:", Counter(y))

Original dataset shape: Counter({0: 5073168, 1: 5177})


In [27]:
# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-0.0137455 , -0.01388989, -0.17334858, ..., -0.1871525 ,
         2.75494448,  2.51907519],
       [-0.01389411, -0.01404096, -0.17334858, ..., -0.1871525 ,
        -0.36298372, -0.39697108],
       [-0.01330424, -0.01344133, -0.17334858, ..., -0.1871525 ,
         2.75494448,  2.51907519],
       ...,
       [-0.01389009, -0.01403687,  5.76872333, ..., -0.1871525 ,
        -0.36298372, -0.39697108],
       [-0.01386315, -0.01400949,  5.76872333, ..., -0.1871525 ,
        -0.36298372,  2.51907519],
       [-0.01366686, -0.01380994,  5.76872333, ..., -0.1871525 ,
        -0.36298372, -0.39697108]], shape=(5078345, 38))

In [28]:
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df

,Amount_Received_USD,Amount_Paid_USD,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,Receiving Currency_Canadian Dollar,Receiving Currency_Euro,Receiving Currency_Mexican Peso,Receiving Currency_Ruble,Receiving Currency_Rupee,Receiving Currency_Saudi Riyal,...,Payment Currency_Yen,Payment Currency_Yuan,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,-0.013746,-0.013890,-0.173349,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,-0.172103,-0.32712,-0.761619,-0.593645,3.091385,-0.187152,2.754944,2.519075
1,-0.013894,-0.014041,-0.173349,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,-0.172103,-0.32712,1.312993,-0.593645,-0.323480,-0.187152,-0.362984,-0.396971
2,-0.013304,-0.013441,-0.173349,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,-0.172103,-0.32712,-0.761619,-0.593645,3.091385,-0.187152,2.754944,2.519075
3,-0.013781,-0.013926,-0.173349,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,-0.172103,-0.32712,-0.761619,-0.593645,3.091385,-0.187152,2.754944,2.519075
4,-0.012420,-0.012542,-0.173349,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,-0.172103,-0.32712,-0.761619,-0.593645,3.091385,-0.187152,2.754944,2.519075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,-0.013769,-0.013914,5.768723,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,5.810466,-0.32712,-0.761619,-0.593645,-0.323480,-0.187152,-0.362984,-0.396971
5078341,-0.013807,-0.013952,5.768723,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,5.810466,-0.32712,-0.761619,-0.593645,-0.323480,-0.187152,-0.362984,-0.396971
5078342,-0.013890,-0.014037,5.768723,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,5.810466,-0.32712,-0.761619,-0.593645,-0.323480,-0.187152,-0.362984,-0.396971
5078343,-0.013863,-0.014009,5.768723,-0.119538,-0.169211,-0.54775,-0.149506,-0.178823,-0.19826,-0.134299,...,-0.177557,-0.20962,5.810466,-0.32712,-0.761619,-0.593645,-0.323480,-0.187152,-0.362984,2.519075


### Undersampling

In [29]:
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_scaled_df, y)

print("Resampled dataset shape:", Counter(y_resampled))

Resampled dataset shape: Counter({0: 51770, 1: 5177})


In [30]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Calculate Preliminary VIF and OLS MAE

In [31]:
model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train, y_train) 

weights_df = pd.DataFrame({
    'Feature': X.columns,
    'Weight (Coefficient)': model.coef_[0]
}).sort_values(by='Weight (Coefficient)', ascending=False)
print(weights_df) 

                               Feature  Weight (Coefficient)
13        Receiving Currency_US Dollar              0.256971
15             Receiving Currency_Yuan              0.172807
23        Payment Currency_Saudi Riyal              0.155236
5              Receiving Currency_Euro              0.125418
19               Payment Currency_Euro              0.099726
21              Payment Currency_Ruble              0.071600
22              Payment Currency_Rupee              0.064014
25        Payment Currency_Swiss Franc              0.061850
9       Receiving Currency_Saudi Riyal              0.047317
20       Payment Currency_Mexican Peso              0.046692
28                Payment Currency_Yen              0.038779
18    Payment Currency_Canadian Dollar              0.037346
30              Payment Format_Bitcoin              0.033602
16            Payment Currency_Bitcoin              0.033490
17        Payment Currency_Brazil Real              0.024196
14              Receivin

In [32]:
y_pred = results.predict(X_test)

In [33]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.1431655933601344

In [34]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled.columns
vif_data["VIF"] = [variance_inflation_factor(X_resampled.values, i) for i in range(X_resampled.shape[1])]

In [35]:
vif_data

,Feature,VIF
0,Amount_Received_USD,6.923112e+05
1,Amount_Paid_USD,6.923109e+05
2,Receiving Currency_Bitcoin,1.218499e+02
3,Receiving Currency_Brazil Real,8.010165e+01
4,Receiving Currency_Canadian Dollar,9.729818e+01
5,Receiving Currency_Euro,3.943032e+02
6,Receiving Currency_Mexican Peso,1.136049e+02
7,Receiving Currency_Ruble,1.132333e+02
8,Receiving Currency_Rupee,1.160547e+02
9,Receiving Currency_Saudi Riyal,1.290396e+02


In [36]:
len(vif_data.sort_values(by="VIF")[vif_data['VIF'] > 5])

C:\Users\caleb\AppData\Local\Temp\ipykernel_26456\2692098991.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  len(vif_data.sort_values(by="VIF")[vif_data['VIF'] > 5])


34

In [37]:
len(X_resampled.columns)

38

The inf on Bitcoin & Ruble:

Bitcoin: Payment Format_Bitcoin, Payment Currency_Bitcoin, and Receiving Currency_Bitcoin always happen at the exact same time. If someone pays via Bitcoin format, the currency is always Bitcoin. They are 100% identical vectors.

Ruble: Receiving Currency_Ruble and Payment Currency_Ruble are perfectly correlated, meaning every Ruble transaction in your dataset has the exact same sender and receiver currency. This is a limitation of the sample size or the dataset creation

### Dropping Columns with high VIF 

In [38]:
# 1. Break the Amount Mirror: Keep only ONE transactional volume metric
# Drop 'Amount_Paid_USD' since it's identical to 'Amount_Received_USD'
if "Amount_Paid_USD" in X_resampled.columns:
    X_resampled = X_resampled.drop(columns=["Amount_Paid_USD"])
    print("Dropped: Amount_Paid_USD (Redundant with Amount_Received_USD)")

# 2. Break Cross-Variable Identity Traps (Bitcoin & Ruble overlap)
# Bitcoin format dictates Bitcoin currency. We only need one indicator.
cols_to_remove = [
    "Payment Format_Bitcoin",
    "Payment Currency_Bitcoin",
    "Payment Currency_Ruble",
]

X_resampled = X_resampled.drop(columns=[c for c in cols_to_remove if c in X_resampled.columns])
print(f"Dropped overlapping cross-variables: {cols_to_remove}")


# Identify all remaining currency features
recv_cols = [c for c in X_resampled.columns if c.startswith("Receiving Currency_")]
pay_cols = [c for c in X_resampled.columns if c.startswith("Payment Currency_")]

# Extract clean currency names present in both sets
# e.g., 'US Dollar', 'Euro'
currencies = [c.split("_")[1] for c in pay_cols]

# Reconstruct if a transaction required a Foreign Exchange (FX) conversion
# It's an FX transaction if Payment Currency != Receiving Currency
# We look across all currencies to find where they mismatch
fx_conditions = []
for curr in currencies:
    p_col = f"Payment Currency_{curr}"
    r_col = f"Receiving Currency_{curr}"

    if p_col in X_resampled.columns and r_col in X_resampled.columns:
        # If one is 1 and the other is 0, it's an FX transaction
        fx_conditions.append(X_resampled[p_col] != X_resampled[r_col])

# If any currency pair mismatched, Is_FX becomes 1
if fx_conditions:
    X_resampled["Is_FX"] = np.amax(np.column_stack(fx_conditions), axis=1).astype(int)
    print("Created feature: Is_FX")

# Drop the redundant set of currency columns entirely
# Dropping 'Receiving Currency' eliminates the ~170 VIF inflation
X_resampled = X_resampled.drop(columns=recv_cols)
print(f"Dropped redundant tracking set: {recv_cols}")

# Reset the constant properly to ensure accurate VIF scores
if "const" in X_resampled.columns:
    X_resampled = X_resampled.drop(columns=["const"])
X_resampled.insert(0, "const", 1.0)

# Recalculate clean VIF Data
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_resampled.values, i) for i in range(X_resampled.shape[1])
]

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


Dropped: Amount_Paid_USD (Redundant with Amount_Received_USD)
Dropped overlapping cross-variables: ['Payment Format_Bitcoin', 'Payment Currency_Bitcoin', 'Payment Currency_Ruble']
Created feature: Is_FX
Dropped redundant tracking set: ['Receiving Currency_Bitcoin', 'Receiving Currency_Brazil Real', 'Receiving Currency_Canadian Dollar', 'Receiving Currency_Euro', 'Receiving Currency_Mexican Peso', 'Receiving Currency_Ruble', 'Receiving Currency_Rupee', 'Receiving Currency_Saudi Riyal', 'Receiving Currency_Shekel', 'Receiving Currency_Swiss Franc', 'Receiving Currency_UK Pound', 'Receiving Currency_US Dollar', 'Receiving Currency_Yen', 'Receiving Currency_Yuan']


In [39]:
vif_data

,Feature,VIF
0,const,0.000000
1,Amount_Received_USD,1.000866
2,Payment Currency_Brazil Real,1.153778
3,Payment Currency_Canadian Dollar,1.297228
4,Payment Currency_Euro,2.911244
5,Payment Currency_Mexican Peso,1.220964
6,Payment Currency_Rupee,1.388162
7,Payment Currency_Saudi Riyal,1.232757
8,Payment Currency_Shekel,1.372433
9,Payment Currency_Swiss Franc,1.479148


In [40]:
len(vif_data)

22

In [41]:
vif_data = vif_data[vif_data['VIF'] < 5]
len(vif_data)

20

In [42]:
vif_data

,Feature,VIF
0,const,0.000000
1,Amount_Received_USD,1.000866
2,Payment Currency_Brazil Real,1.153778
3,Payment Currency_Canadian Dollar,1.297228
4,Payment Currency_Euro,2.911244
5,Payment Currency_Mexican Peso,1.220964
6,Payment Currency_Rupee,1.388162
7,Payment Currency_Saudi Riyal,1.232757
8,Payment Currency_Shekel,1.372433
9,Payment Currency_Swiss Franc,1.479148


In [43]:
model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train, y_train) 

weights_df = pd.DataFrame({
    'Feature': X.columns,
    'Weight (Coefficient)': model.coef_[0]
}).sort_values(by='Weight (Coefficient)', ascending=False)
print(weights_df) 

                               Feature  Weight (Coefficient)
13        Receiving Currency_US Dollar              0.256971
15             Receiving Currency_Yuan              0.172807
23        Payment Currency_Saudi Riyal              0.155236
5              Receiving Currency_Euro              0.125418
19               Payment Currency_Euro              0.099726
21              Payment Currency_Ruble              0.071600
22              Payment Currency_Rupee              0.064014
25        Payment Currency_Swiss Franc              0.061850
9       Receiving Currency_Saudi Riyal              0.047317
20       Payment Currency_Mexican Peso              0.046692
28                Payment Currency_Yen              0.038779
18    Payment Currency_Canadian Dollar              0.037346
30              Payment Format_Bitcoin              0.033602
16            Payment Currency_Bitcoin              0.033490
17        Payment Currency_Brazil Real              0.024196
14              Receivin

In [44]:
y_pred = results.predict(X_test)

In [45]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.1431655933601344

Attempting Polynomial Terms

In [46]:
X_train_with_square = pd.concat((X_train, (X_train**2).rename(columns = {f"{var}": f"{var}_2" for var in X.columns})), axis = 1) 
X_test_with_square = pd.concat((X_test, (X_test**2).rename(columns = {f"{var}": f"{var}_2" for var in X.columns})), axis = 1) 


model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train_with_square, y_train) 

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [47]:
y_pred = results.predict(X_test_with_square)

In [48]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.1440033248860868

In [49]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_train_with_square.columns
vif_data["VIF"] = [variance_inflation_factor(X_train_with_square.values, i) for i in range(X_train_with_square.shape[1])]

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [50]:
len(vif_data.sort_values(by="VIF")[vif_data['VIF'] > 5])

C:\Users\caleb\AppData\Local\Temp\ipykernel_26456\2692098991.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  len(vif_data.sort_values(by="VIF")[vif_data['VIF'] > 5])


76

X of the features have VIF > 5, showing multicollinearity

## PPP

### Read Data and Perform Undersampling

In [51]:
# Import credit card datasaet
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ppp_cleaned.csv')
df

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,48.0,2.0,24,769358.78,769358.78,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12409.01,0,1588291200,1605830400,1608249600
1,0.0,48.0,2.0,24,736927.79,736927.79,11.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10094.90,0,1588291200,1628726400,1632787200
2,0.0,48.0,2.0,24,691355.00,691355.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9218.07,0,1588291200,1612915200,1615939200
3,0.0,48.0,2.0,24,499871.00,499871.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23803.38,0,1588291200,1631232000,1634342400
4,0.0,48.0,2.0,24,367437.00,367437.00,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,14697.48,0,1588291200,1617840000,1629158400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968527,0.0,56.0,2.0,24,150000.00,150000.00,54.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10000.00,0,1585872000,1607472000,1610496000
968528,0.0,56.0,2.0,24,150000.00,150000.00,31.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3452.38,0,1586822400,1604361600,1607385600
968529,1.0,56.0,2.0,60,150000.00,150000.00,54.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,29999.40,0,1613088000,1629158400,1631664000
968530,0.0,56.0,2.0,60,150000.00,150000.00,18.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21428.57,0,1586908800,1645574400,1646697600


First lets undersample since this is a fraud dataset

In [52]:
X = df.drop(columns='Fraud')

In [53]:
y = df['Fraud']

In [54]:
print("Original dataset shape:", Counter(y))

Original dataset shape: Counter({0: 968437, 1: 95})


In [55]:
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Resampled dataset shape: Counter({0: 950, 1: 95})


In [56]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Model and VIF

In [57]:
model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train, y_train) 

weights_df = pd.DataFrame({
    'Feature': X.columns,
    'Weight (Coefficient)': model.coef_[0]
}).sort_values(by='Weight (Coefficient)', ascending=False)
print(weights_df) 

                          Feature  Weight (Coefficient)
37                PROCEED_Per_Job          3.166719e-05
16                   RENT_PROCEED          7.231936e-06
4           InitialApprovalAmount          1.324364e-06
13              UTILITIES_PROCEED          1.175274e-06
14                PAYROLL_PROCEED          8.056119e-07
1                   BorrowerState          6.343110e-08
11                   ProjectState          6.343110e-08
40             LoanStatusDate_int          3.836646e-09
20                   BusinessType          3.637350e-09
10         BusinessAgeDescription          3.600933e-09
22                        Veteran          1.006619e-09
7             RuralUrbanIndicator          5.080748e-10
23                      NonProfit          1.045688e-10
27             ForgivenPercentage          4.199410e-11
31            PAYROLL_PROCEED_pct          2.512750e-11
33               RENT_PROCEED_pct          2.062831e-11
30          UTILITIES_PROCEED_pct          1.867

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [58]:
y_pred = results.predict(X_test)

In [59]:
average_precision_score(y_true=y_test, y_score=y_pred)

0.2006911217437533

In [60]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled.columns
vif_data["VIF"] = [variance_inflation_factor(X_resampled.values, i) for i in range(X_resampled.shape[1])]

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


In [61]:
vif_data

,Feature,VIF
0,ProcessingMethod,1.018787e+01
1,BorrowerState,inf
2,LoanStatus,6.484727e+00
3,Term,5.669173e+00
4,InitialApprovalAmount,inf
5,CurrentApprovalAmount,inf
6,ServicingLenderState,1.812911e+01
7,RuralUrbanIndicator,1.075239e+00
8,HubzoneIndicator,1.430930e+00
9,LMIIndicator,1.380365e+00


In [62]:
len(vif_data.sort_values(by="VIF")[vif_data['VIF'] > 5])

C:\Users\caleb\AppData\Local\Temp\ipykernel_26456\2692098991.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  len(vif_data.sort_values(by="VIF")[vif_data['VIF'] > 5])


31

### Removing Multicollinearity

In [63]:
cols_to_drop = ['TOTAL_PROCEED', 'ApprovalDiff', 'NotForgivenAmount']
X_resampled_reduced = X_resampled.drop(columns=cols_to_drop)

In [64]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled_reduced.columns
vif_data["VIF"] = [variance_inflation_factor(X_resampled_reduced.values, i) for i in range(X_resampled_reduced.shape[1])]
vif_data

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Feature,VIF
0,ProcessingMethod,1.018787e+01
1,BorrowerState,inf
2,LoanStatus,6.484727e+00
3,Term,5.669173e+00
4,InitialApprovalAmount,4.654207e+01
5,CurrentApprovalAmount,inf
6,ServicingLenderState,1.812911e+01
7,RuralUrbanIndicator,1.075239e+00
8,HubzoneIndicator,1.430930e+00
9,LMIIndicator,1.380365e+00


We still have issues with: 

BorrowerState vs ProjectState

CurrentApprovalAmount vs PROCEED columns

In [65]:
X_resampled_reduced = X_resampled_reduced.drop(columns=['ProjectState', 'CurrentApprovalAmount'])
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled_reduced.columns
vif_data["VIF"] = [variance_inflation_factor(X_resampled_reduced.values, i) for i in range(X_resampled_reduced.shape[1])]
vif_data

,Feature,VIF
0,ProcessingMethod,10.187875
1,BorrowerState,1.325390
2,LoanStatus,6.484727
3,Term,5.669173
4,InitialApprovalAmount,46.542070
5,ServicingLenderState,18.129114
6,RuralUrbanIndicator,1.075239
7,HubzoneIndicator,1.430930
8,LMIIndicator,1.380365
9,BusinessAgeDescription,1.104773


We no longer have inf values, but there is still extreme multicollinearity

In [66]:
# Dropping dollar amounts and keeping percents
raw_proceed_cols = [
    'UTILITIES_PROCEED', 'PAYROLL_PROCEED', 'MORTGAGE_INTEREST_PROCEED', 
    'RENT_PROCEED', 'REFINANCE_EIDL_PROCEED', 'HEALTH_CARE_PROCEED', 'DEBT_INTEREST_PROCEED'
]
X_resampled_final = X_resampled_reduced.drop(columns=raw_proceed_cols)

In [67]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled_final.columns
vif_data["VIF"] = [variance_inflation_factor(X_resampled_final.values, i) for i in range(X_resampled_final.shape[1])]
vif_data

,Feature,VIF
0,ProcessingMethod,10.180349
1,BorrowerState,1.319581
2,LoanStatus,6.458122
3,Term,5.661059
4,InitialApprovalAmount,5.980131
5,ServicingLenderState,18.088262
6,RuralUrbanIndicator,1.072175
7,HubzoneIndicator,1.421331
8,LMIIndicator,1.372516
9,BusinessAgeDescription,1.089721


In [68]:
# The percents add up to 1, so we can easily predict 1 given the others. Going to remove one 
X_resampled_final.drop(columns='PAYROLL_PROCEED_pct', inplace=True)

In [69]:
vif_data = pd.DataFrame()
vif_data["Feature"] = X_resampled_final.columns
vif_data["VIF"] = [variance_inflation_factor(X_resampled_final.values, i) for i in range(X_resampled_final.shape[1])]
vif_data

,Feature,VIF
0,ProcessingMethod,5.828360
1,BorrowerState,4.481083
2,LoanStatus,134.257033
3,Term,19.893624
4,InitialApprovalAmount,9.225340
5,ServicingLenderState,87.380572
6,RuralUrbanIndicator,8.855137
7,HubzoneIndicator,1.908323
8,LMIIndicator,1.850449
9,BusinessAgeDescription,4.108318


In [70]:
# Drop cols greater than 5
vif_data = vif_data[vif_data['VIF'] < 5]
X_resampled_final_final = X_resampled_final[vif_data['Feature'].values]

In [71]:
final_cols = X_resampled_final_final.columns
X_train = X_train[final_cols]
X_test = X_test[final_cols]

model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train, y_train) 


In [72]:
len(X_resampled_final_final.columns)

15

In [73]:
y_pred = results.predict(X_test)
average_precision_score(y_true=y_test, y_score=y_pred)

0.09090909090909091

### Polynomial Terms

In [74]:
X_train = X_train[final_cols]
X_test = X_test[final_cols]

X_train_with_square = pd.concat((X_train, (X_train**2).rename(columns = {f"{var}": f"{var}_2" for var in X_resampled_final_final.columns})), axis = 1) 
X_test_with_square = pd.concat((X_test, (X_test**2).rename(columns = {f"{var}": f"{var}_2" for var in X_resampled_final_final.columns})), axis = 1) 


model = LogisticRegression(max_iter=2000, random_state=random_state)
results = model.fit(X_train_with_square, y_train) 


In [75]:
y_pred = results.predict(X_test_with_square)
average_precision_score(y_true=y_test, y_score=y_pred)

0.09090909090909091

This model is actually a bit worse